In [ ]:
from statistics import mode

from pandas.core.computation.expressions import where

"""
Exercise 4: NumPy Array Operations
Complete the following tasks using NumPy.
"""

In [ ]:
import numpy as np

# Task 1: Create a 5x5 matrix where border elements are 1 and interior is 0

# Create matrix
matrix55 = np.zeros((5, 5))

# Now make border items 1
matrix55[:, 0] = 1
matrix55[:, -1] = 1

matrix55

In [ ]:
# Task 2: Normalize a random array

# Each col having mean = 0 & std = 1 by default
np.random.seed(42)
randArr = np.random.randn(100, 3)
randArr


In [32]:
# Task 3: Implement linear regression solution using normal equation

X = np.random.randn(50, 3)
true_theta = np.array([2.5, -1.2, 3.7])
y = X @ true_theta + np.random.randn(50) * 0.1

# Normal eqn (theta = (X^T X)^(-1) X^T y) translated:
theta_hat = np.linalg.inv((np.transpose(X) @ X)) @ (np.transpose(X) @ y)

print(f"True Theta: ", true_theta)
print(f"Theta hat: ", theta_hat)


True Theta:  [ 2.5 -1.2  3.7]
Theta hat:  [ 2.51723721 -1.19783796  3.72399266]


In [ ]:
"""
Exercise 5: Pandas Data Analysis
Analyze a dataset of student performance.
"""

import pandas as pd
import numpy as np

# Create sample dataset
np.random.seed(42)
n_students = 200

data = {
    'student_id': range(1000, 1000 + n_students),
    'major': np.random.choice(['CS', 'Math', 'Physics', 'Biology'], n_students),
    'year': np.random.choice([1, 2, 3, 4], n_students),
    'exam_score': np.random.normal(75, 10, n_students).clip(0, 100),
    'assignments_completed': np.random.randint(0, 11, n_students),
    'hours_studied': np.random.normal(15, 5, n_students).clip(1, 40)
}

df = pd.DataFrame(data)

# Introduce some NaN values
df.loc[np.random.choice(n_students, 10), 'exam_score'] = np.nan
df.loc[np.random.choice(n_students, 5), 'hours_studied'] = np.nan

In [ ]:
## Task 1: Data Cleaning and Exploration

# Display basic information about the dataset
df.describe()


# Identify and count missing values
missingValsCount = df.isnull().sum().sum()
print(f"Total missing values in data frame: ", missingValsCount)


# Fill missing exam_score with the mean score for the student's major
df_stats = df.groupby('major')['exam_score'].agg(['mean'])

for index, row in df[df['exam_score'].isnull()].iterrows():
    major = row['major']
    mean_for_major = df_stats.loc[major, 'mean']
    df.loc[index, 'exam_score'] = mean_for_major


# Fill missing hours_studied with the median for the student's year
df_stats2 = df.groupby('year')['hours_studied'].agg(['median'])
df_nulls2 = df[df['hours_studied'].isnull()]


for index, row in df[df['hours_studied'].isnull()].iterrows():
    year = row['year']
    median_for_year = df_stats2.loc[year, 'median']
    df.loc[index, 'hours_studied'] = median_for_year


In [ ]:
## Task 2: Analysis

# Calculate and display the average exam_score by major
avg_score_by_major = df.groupby('major')['exam_score'].agg(['mean'])
avg_score_by_major.head()


# Find the major with the highest average exam_score
avg_score_by_major.sort_values('mean', ascending=False).head(1)


# Calculate the correlation between hours_studied and exam_score
df_correlation = df['exam_score'].corr(df['hours_studied'])
print(f"Correlation b/n exam scores and hours studied is: ", df_correlation)


# Create new 'performance' column
# 'Excellent' (>90), 'Good' (80-90), 'Average' (70-80), 'Needs Improvement' (<70)
df['Performance'] = np.where(df['exam_score'] > 90, 'Excellent',
                    np.where((df['exam_score'] >= 80) & (df['exam_score'] < 90), 'Good',
                    np.where((df['exam_score'] >= 70) & (df['exam_score'] < 80), 'Average', 'Needs Improvement')))

In [59]:
# Task 3: Advanced Analysis

# Task 3: Advanced Analysis (10 points)
# TODO: For each major and year combination, calculate:
#       - Number of students
#       - Average exam score
#       - Average hours studied

df.groupby(['major', 'year']).agg({ 'exam_score': ['mean'], 'hours_studied': ['mean'], 'student_id': ['count'] })


# TODO: Identify top 5 students based on exam_score (handle ties appropriately)
df['Rank'] = df['exam_score'].rank(method='dense', ascending=False)
leaderboard = df[df['Rank'] <= 5].sort_values(by='Rank')


# TODO: Create a pivot table showing average exam_score by major (rows) and year (columns)
table = pd.pivot_table(df, values='exam_score', index=['major'], columns=['year'], aggfunc='mean')




year,1,2,3,4
major,,,,
Biology,76.570059,74.899613,80.178476,70.697689
CS,77.098194,76.155936,72.347626,78.119997
Math,74.053167,81.226714,72.017537,73.103066
Physics,77.828686,73.277734,73.134319,78.130194
